In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_smartdata")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def fill_NA_null_value(p_value):
    if p_value is None:
        return "NA"
    else:
        return p_value
    

In [0]:
fill_na_udf = F.udf(fill_NA_null_value, StringType())

In [0]:
df_partners = spark.table(f"{catalogo}.{esquema_source}.partners")
df_sales = spark.table(f"{catalogo}.{esquema_source}.sales")

In [0]:
df_partners = df_partners.dropna(how="all")\
                        .filter((col("partner_id").isNotNull()) | (col("create_date")).isNotNull())

df_sales = df_sales.dropna(how="all")\
                    .filter((col("sales_id").isNotNull()) | (col("product_id_id")).isNotNull())

In [0]:
df_partners = df_partners.withColumn("nit_new", fill_na_udf("nit")) \
    .withColumn("par_email_new", fill_na_udf("par_email")) \
    .withColumn("mobile_new", fill_na_udf("mobile")) \
    .withColumnRenamed("nit_name_new", "nit_name")


In [0]:
df_partners.display()

In [0]:
#df_updated = df_partners.filter(col("partner_id").isNotNull())
df_updated = df_partners.where(col("partner_id") > 1).select("*")                                  


In [0]:
df_updated.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.partners_transformed")

In [0]:
%sql
select * from catalog_smartdata.silver.partners_transformed

In [0]:
%sql
select * from catalog_smartdata.bronze.sales

In [0]:
display(_sqldf)

In [0]:
# fill campos nullos
df_sales = _sqldf.where(col("sales_id") > 1).select("*")                                    

In [0]:
df_sales.display()

In [0]:
df_products_transformed = df_sales.groupBy(col("product_id"), col("product_code")).agg(
                                                     count(col("product_id")).alias("conteo")
                                                     ).orderBy(col("product_id")
                                                     )

In [0]:
df_products_transformed.display()
df_products_transformed.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.products_transformed")

In [0]:
# save the sales table
df_sales.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.sales_transformed")